# Pooled FastText features: LR vs LightGBM (RQ2)

FastText `wiki-news-300d-1M` (frozen) pooled into one vector per comment, then logistic regression or LightGBM, trained and evaluated on the shared 5 stratified folds with `comment_heavy` as input.

In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()

if IN_COLAB or IN_KAGGLE:
    if IN_KAGGLE:
        %cd /kaggle/working
    !git clone -q https://github.com/mnhhuyen/AI4SE_toxic_comments.git
    %cd AI4SE_toxic_comments
    !pip install -q -e ".[dev]" lightgbm tabulate

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# Kaggle: attach the Jigsaw competition data, optionally a dataset with wiki-news-300d-1M.vec
FASTTEXT = ROOT / "data" / "embeddings" / "fasttext" / "wiki-news-300d-1M.vec"
RAW_TRAIN = ROOT / "data" / "raw" / "train.csv"
if IN_KAGGLE:
    import pandas as pd

    attached = next(Path("/kaggle/input").rglob("wiki-news-300d-1M.vec"), None)
    if attached and not FASTTEXT.exists():
        FASTTEXT.symlink_to(attached)
    if not RAW_TRAIN.exists():
        RAW_TRAIN.parent.mkdir(parents=True, exist_ok=True)
        pd.read_csv(next(Path("/kaggle/input").rglob("train.csv*"))).to_csv(RAW_TRAIN, index=False)

if (IN_COLAB or IN_KAGGLE) and not FASTTEXT.exists():
    !wget -q -c https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip -P {FASTTEXT.parent}
    !unzip -q -n {FASTTEXT.parent}/wiki-news-300d-1M.vec.zip -d {FASTTEXT.parent}

/content/AI4SE_toxic_comments
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for toxic-comments (pyproject.toml) ... done


## Data and folds

In [2]:
import pandas as pd

from toxic_comments.cleaning import process_cleaning, validate_training_data
from toxic_comments.config import (ID_COLUMN, K_FOLD_DATA_DIR, LABEL_COLUMNS, MODELS_DIR,
                                   RAW_DATA_DIR, RESULTS_DIR)
from toxic_comments.splits import save_kfold_datasets

TEXT_COLUMNS = ["comment_text", "comment_light", "comment_heavy"]

if not list(K_FOLD_DATA_DIR.glob("fold_*/test.csv")):
    data = process_cleaning(validate_training_data(pd.read_csv(RAW_DATA_DIR / "train.csv")))
    data[TEXT_COLUMNS] = data[TEXT_COLUMNS].fillna("").astype(str)
    data = data[data["is_empty_heavy"] == 0].reset_index(drop=True)
    save_kfold_datasets(data[[ID_COLUMN, *TEXT_COLUMNS, *LABEL_COLUMNS]], n_splits=5, strategy="stratified")

sorted(path.name for path in K_FOLD_DATA_DIR.glob("fold_*"))


>>> dataset (159,571 rows)
    cleaning level 1 (light) ...
    cleaning level 2 (heavy) ...
    75 rows empty in HEAVY but fine in LIGHT (non-Latin / emoji) -> KEPT, flagged as is_non_latin
    dropped 95 rows with no usable text at all
    170 rows unusable for bag-of-words -> filter on is_empty_heavy for TF-IDF


['fold_1', 'fold_2', 'fold_3', 'fold_4', 'fold_5']

## Training

`POOLING` is the configuration selected in RQ1 (FastText + max + supervised attention). Results go to `results/pooled/<pooling>/` so runs with different poolings do not overwrite each other.

Per fold, the embeddings step is fitted once and its features are shared by LR and LightGBM: tokenizing (CPU), then attention training and pooling (GPU), then the classifiers (CPU).

In [3]:
import torch

from toxic_comments.train import train_pooled_from_folds

POOLING = "max_supervised_attention"
MODELS = ["fasttext_logistic_regression", "fasttext_lightgbm"]
RUN_RESULTS_DIR = RESULTS_DIR / "pooled" / POOLING
RUN_MODELS_DIR = MODELS_DIR / "pooled" / POOLING

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
pending = [name for name in MODELS if not (RUN_RESULTS_DIR / f"{name}_fold_metrics.csv").exists()]
print("skipping (results found):", [name for name in MODELS if name not in pending])
if pending:
    train_pooled_from_folds(folds_dir=K_FOLD_DATA_DIR, model_names=pending, output_dir=RUN_MODELS_DIR,
                            results_dir=RUN_RESULTS_DIR, embedding_pooling=POOLING)

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


GPU: none
skipping (results found): []


folds:   0%|          | 0/5 [00:00<?, ?fold/s]

[fold_1] embeddings...


load wiki-news-300d-1M.vec: 0 lines [00:00, ? lines/s]

tokenize:   0%|          | 0/127520 [00:00<?, ?text/s]

... (55 outputs left out)

pooling (cpu):   0%|          | 0/32 [00:00<?, ?it/s]

[fold_5] embeddings done in 357.7s
[fold_5] fit fasttext_logistic_regression...
[fold_5] fit fasttext_logistic_regression done in 136.2s
[fold_5] fit fasttext_lightgbm...


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[fold_5] fit fasttext_lightgbm done in 208.1s


## Results (mean ± std over 5 folds)

In [4]:
METRICS = ["macro_roc_auc", "micro_roc_auc", "micro_f1", "macro_f1",
           "micro_precision", "micro_recall", "subset_accuracy", "hamming_loss"]

folds = pd.concat([pd.read_csv(RUN_RESULTS_DIR / f"{name}_fold_metrics.csv") for name in MODELS])
stats = folds.groupby("model_name")[METRICS].agg(["mean", "std"]).loc[MODELS]
summary = pd.DataFrame({
    metric: stats[(metric, "mean")].map("{:.4f}".format) + " ± " + stats[(metric, "std")].map("{:.4f}".format)
    for metric in METRICS
}).T
print(summary.to_markdown())

|                 | fasttext_logistic_regression   | fasttext_lightgbm   |
|:----------------|:-------------------------------|:--------------------|
| macro_roc_auc   | 0.9715 ± 0.0009                | 0.9800 ± 0.0023     |
| micro_roc_auc   | 0.9801 ± 0.0002                | 0.9830 ± 0.0008     |
| micro_f1        | 0.5722 ± 0.0026                | 0.7347 ± 0.0042     |
| macro_f1        | 0.4428 ± 0.0027                | 0.5381 ± 0.0051     |
| micro_precision | 0.4156 ± 0.0026                | 0.8073 ± 0.0045     |
| micro_recall    | 0.9185 ± 0.0041                | 0.6742 ± 0.0067     |
| subset_accuracy | 0.8341 ± 0.0015                | 0.9193 ± 0.0008     |
| hamming_loss    | 0.0504 ± 0.0005                | 0.0179 ± 0.0002     |


## Trainable parameters

In [5]:
import joblib

fitted = {name: joblib.load(RUN_MODELS_DIR / "fold_1" / f"{name}.joblib") for name in MODELS}

vectorizer = fitted["fasttext_logistic_regression"].named_steps["embeddings"]
attention_params = sum(getattr(vectorizer, attr).size
                       for attr in ["attention_W_", "attention_b_", "attention_v_"]
                       if hasattr(vectorizer, attr))
lr = fitted["fasttext_logistic_regression"].named_steps["classifier"]
lr_params = sum(est.coef_.size + est.intercept_.size for est in lr.estimators_)
trees = sum(est.n_estimators_ for est in fitted["fasttext_lightgbm"].named_steps["classifier"].estimators_)

print(f"pooled dim: {vectorizer.output_dim_}")
print(f"attention params: {attention_params:,}")
print(f"LR: {lr_params:,} + attention = {lr_params + attention_params:,}")
print(f"LightGBM: {trees} trees + {attention_params:,} attention params")

pooled dim: 600
attention params: 90,600
LR: 3,606 + attention = 94,206
LightGBM: 1800 trees + 90,600 attention params


## LaTeX rows for `tab:rq2-overall`

In [6]:
TABLE_METRICS = ["macro_roc_auc", "micro_roc_auc", "micro_f1", "macro_f1"]
LABELS = {"fasttext_logistic_regression": "LR", "fasttext_lightgbm": "LightGBM"}

for name in MODELS:
    cells = " & ".join(f"{stats.loc[name, (m, 'mean')]:.4f} $\\pm$ {stats.loc[name, (m, 'std')]:.4f}"
                       for m in TABLE_METRICS)
    print(f"Pooled features & {LABELS[name]} & ... & {cells} \\\\")

Pooled features & LR & ... & 0.9715 $\pm$ 0.0009 & 0.9801 $\pm$ 0.0002 & 0.5722 $\pm$ 0.0026 & 0.4428 $\pm$ 0.0027 \\
Pooled features & LightGBM & ... & 0.9800 $\pm$ 0.0023 & 0.9830 $\pm$ 0.0008 & 0.7347 $\pm$ 0.0042 & 0.5381 $\pm$ 0.0051 \\


## Per-label results

In [7]:
per_label = pd.concat([pd.read_csv(RUN_RESULTS_DIR / f"{name}_fold_per_label.csv") for name in MODELS])

for metric in ["roc_auc", "f1", "precision", "recall"]:
    table = per_label.pivot_table(index="label", columns="model_name", values=metric)[MODELS]
    print(f"\n{metric}\n{table.loc[LABEL_COLUMNS].round(4).to_markdown()}")


roc_auc
| label         |   fasttext_logistic_regression |   fasttext_lightgbm |
|:--------------|-------------------------------:|--------------------:|
| toxic         |                         0.9724 |              0.9739 |
| severe_toxic  |                         0.9779 |              0.9858 |
| obscene       |                         0.9883 |              0.9901 |
| threat        |                         0.945  |              0.9699 |
| insult        |                         0.9794 |              0.9813 |
| identity_hate |                         0.9662 |              0.979  |

f1
| label         |   fasttext_logistic_regression |   fasttext_lightgbm |
|:--------------|-------------------------------:|--------------------:|
| toxic         |                         0.6899 |              0.7799 |
| severe_toxic  |                         0.3159 |              0.2733 |
| obscene       |                         0.6911 |              0.8168 |
| threat        |                     

## Paired comparison with the sequence models

Fold-by-fold difference in macro ROC-AUC against DPCNN and Bi-GRU (same folds).

In [8]:
SEQUENCE_MODELS = ["dpcnn", "bigru_shared_max_attention"]

sequence = pd.concat([pd.read_csv(RESULTS_DIR / f"{name}_fold_metrics.csv") for name in SEQUENCE_MODELS
                      if (RESULTS_DIR / f"{name}_fold_metrics.csv").exists()])
scores = pd.concat([folds, sequence]).pivot_table(index="fold", columns="model_name", values="macro_roc_auc")

for pooled in MODELS:
    for other in [m for m in SEQUENCE_MODELS if m in scores]:
        diff = scores[other] - scores[pooled]
        print(f"{other} - {pooled}: {diff.mean():+.4f} ± {diff.std():.4f}, better on {(diff > 0).sum()}/5 folds")

diff = scores["fasttext_lightgbm"] - scores["fasttext_logistic_regression"]
print(f"LightGBM - LR: {diff.mean():+.4f} ± {diff.std():.4f}, better on {(diff > 0).sum()}/5 folds")

dpcnn - fasttext_logistic_regression: +0.0162 ± 0.0013, better on 5/5 folds
bigru_shared_max_attention - fasttext_logistic_regression: +0.0161 ± 0.0008, better on 5/5 folds
dpcnn - fasttext_lightgbm: +0.0077 ± 0.0018, better on 5/5 folds
bigru_shared_max_attention - fasttext_lightgbm: +0.0077 ± 0.0016, better on 5/5 folds
LightGBM - LR: +0.0085 ± 0.0019, better on 5/5 folds
